# Grid Contingency Analysis
Which single failures hurt the grid? How does a failure spread? 

Import necessary packages.

In [47]:
import pandas as pd
import matplotlib.pyplot as plt
import pandapower as pp
import pandapower.networks as pn
import pandapower.plotting
import pandapower.converter
import pandapower.estimation

In [48]:
# load benchmark IEEE 14-bus network
net = pn.power_system_test_cases.case14()

# run standard AC power flow
pp.runpp(net)

# get all transmission lines, put in array
lines = net.line.index.tolist()
print(f"We will test {len(lines)} transmission lines.")

We will test 15 transmission lines.


In [62]:
# N-0 case (for comparison)
v_min, v_max, load_max = 0.95, 1.10, 100 # establish thresholds
net.line['in_service'] = True # set line to active

# run power flow
pp.runpp(net)
print(net.line['max_i_ka'])
print(net.res_line['loading_percent'].describe())

# extract critical values
base_min_v = net.res_bus['vm_pu'].min()
base_max_v = net.res_bus['vm_pu'].max()
base_ll_max = net.res_line['loading_percent'].max()
base_islanded = int(net.res_bus['vm_pu'].isna().sum())

if base_ll_max > load_max or base_min_v < v_min or base_max_v > v_max or base_islanded > 0:
    status = "Violation"
else:
    status = "Pass"

base_case = pd.DataFrame([{
    "Tripped_Line": "N/A",
    "Status": status,
    "Max_Line_Loading_%": base_ll_max,
    "Min_Bus_V_pu": base_min_v,
    "Max_Bus_V_pu": base_max_v,
    "Islanded_Buses": base_islanded,    
}])

display(base_case)

0        42.339020
1        42.339020
2        42.339020
3        42.339020
4        42.339020
5        42.339020
6        42.339020
7     27479.652235
8     27479.652235
9     27479.652235
10    27479.652235
11    27479.652235
12    27479.652235
13    27479.652235
14    27479.652235
Name: max_i_ka, dtype: float64
count    15.000000
mean      0.357327
std       0.409831
min       0.017055
25%       0.070694
50%       0.180865
75%       0.584899
max       1.507568
Name: loading_percent, dtype: float64


,Tripped_Line,Status,Max_Line_Loading_%,Min_Bus_V_pu,Max_Bus_V_pu,Islanded_Buses
0,N/A,Pass,1.507568,1.01,1.09,0


In [ ]:
# N-1 case
n1_rows = []

for i in lines:
    # simulate a fault at a given transmission line
    net.line.loc[i, 'in_service'] = False

# exception handling to manage the result of Newton-Ralphson method
    try:
        pp.runpp(net) # recalculate power flow

        # net.res_bus for result params
        # check: undervoltage and overvoltage violations
        min_bus_voltage = net.res_bus['vm_pu'].min() # 'vm_pu': voltage magnitude at connection bus
        max_bus_voltage = net.res_bus['vm_pu'].max()

        # check: large and unusual angle deviations
        bus_voltage_angle = net.res_bus['va_degree'] # 'va_degree': voltage angle [deg]

        # active and reactive power
        power_a = net.res_bus['p_mw']
        power_r = net.res_bus['q_mvar']

        # check: thermal overload/voltage sag
        ll_max = net.res_line['loading_percent'].max()

        # check
        n_islanded = int(net.res_bus['vm_pu'].isna().sum()) 

        # extend upper voltage limit (default 1.09)
        if ll_max > load_max or min_bus_voltage < v_min or max_bus_voltage > v_max or n_islanded > 0:
            status = "Violation"
        else:
            status = "Pass"


    except pp.LoadflowNotConverged:
        min_bus_voltage = max_bus_voltage = ll_max = float("nan")
        n_islanded = None
        status = "Non-convergence"

    finally:
        net.line.loc[i, 'in_service'] = True # restore the line

    n1_rows.append({
        "Tripped_Line": i,
        "Status": status,
        "Max_Line_Loading_%": ll_max,
        "Min_Bus_V_pu": min_bus_voltage,
        "Max_Bus_V_pu": max_bus_voltage,
        "Islanded_Buses": n_islanded,

    })

# append results to table
results = pd.DataFrame(n1_rows)
display(results)

,Tripped_Line,Status,Max_Line_Loading_%,Min_Bus_V_pu,Max_Bus_V_pu,Islanded_Buses
0,0,Pass,2.513021,0.993484,1.09,0
1,1,Pass,2.315134,1.006442,1.09,0
2,2,Pass,1.423646,1.010000,1.09,0
3,3,Pass,1.366756,1.007096,1.09,0
4,4,Pass,1.363835,1.010000,1.09,0
5,5,Pass,1.562346,1.010000,1.09,0
6,6,Pass,1.713323,1.010000,1.09,0
7,7,Pass,1.511768,1.010000,1.09,0
8,8,Pass,1.509471,1.010000,1.09,0
9,9,Pass,1.516081,0.997979,1.09,0
